# Token Prediction Stability

How stable are predictions across layers, positions, and perturbations?
Layer commitment, position consistency, token competition, component attribution,
and flip sensitivity.

## Why This Matters

Token prediction stability tracks how individual token representations evolve through the network. Understanding token-level dynamics reveals how context is integrated, when predictions form, and how different positions interact to produce the output.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_prediction_stability import (
    prediction_layer_stability, prediction_position_consistency,
    prediction_token_competition, prediction_component_attribution,
    prediction_flip_sensitivity,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Prediction Layer Stability

When does the model commit to its final prediction?

In [ ]:
result = prediction_layer_stability(model, tokens)
early = 'EARLY' if result['is_early_commit'] else 'late'
print(f"Final: token {result['final_prediction']}, commits at layer {result['commit_layer']} [{early}]\n")
for entry in result['per_layer']:
    match = '✓' if entry['matches_final'] else '✗'
    print(f"  Layer {entry['layer']}: pred={entry['prediction']:3d}, "
          f"conf={entry['confidence']:.4f} [{match}]")

## Position Consistency

Is confidence uniform across positions, or do some positions
stand out as hard/easy?

In [ ]:
result = prediction_position_consistency(model, tokens)
unif = 'UNIFORM' if result['is_uniform'] else 'variable'
print(f"Mean conf: {result['mean_confidence']:.4f} ± {result['std_confidence']:.4f} [{unif}]\n")
for p in result['per_position']:
    bar = '█' * int(p['confidence'] * 20)
    print(f"  Pos {p['position']}: pred={p['top_prediction']:3d}, "
          f"conf={p['confidence']:.4f}, "
          f"entropy={p['entropy']:.4f} {bar}")

## Token Competition

Track the top-k candidate tokens through layers.

In [ ]:
result = prediction_token_competition(model, tokens, top_k=4)
decisive = 'DECISIVE' if result['is_decisive'] else 'close'
print(f"Margin: {result['margin']:.4f} [{decisive}]\n")
for t_info in result['per_token']:
    print(f"Token {t_info['token']} (final rank {t_info['final_rank']}, "
          f"prob={t_info['final_probability']:.4f}):")
    for entry in t_info['per_layer']:
        bar = '█' * int(entry['probability'] * 20)
        print(f"    L{entry['layer']}: logit={entry['logit']:+.4f}, "
              f"rank={entry['rank']}, prob={entry['probability']:.4f} {bar}")
    print()

## Component Attribution

Which components contribute most to the winning prediction?

In [ ]:
result = prediction_component_attribution(model, tokens)
print(f"Target: token {result['target_token']} (logit={result['target_logit']:.4f})")
print(f"Top component: {result['top_component']}\n")
for c in result['components']:
    bar = '█' * int(min(abs(c['logit_contribution']), 2) * 10)
    sign = '+' if c['logit_contribution'] > 0 else '-'
    print(f"  {c['component']:10s}: {c['logit_contribution']:+.4f} [{sign}] {bar}")

## Flip Sensitivity

How much noise does it take to flip the prediction?

In [ ]:
result = prediction_flip_sensitivity(model, tokens)
robust = 'ROBUST' if result['is_robust'] else 'fragile'
flip_str = f"at {result['first_flip_noise']}" if result['first_flip_noise'] else 'never'
print(f"Clean: token {result['clean_prediction']} (conf={result['clean_confidence']:.4f})")
print(f"Flips: {flip_str} [{robust}]\n")
for entry in result['per_noise_level']:
    status = 'FLIPPED' if entry['flipped'] else 'stable'
    print(f"  noise={entry['noise_fraction']:.0%}: pred={entry['prediction']:3d}, "
          f"conf={entry['confidence']:.4f} [{status}]")